# Dirty Data Generator.1.0

Dataset generator works in 3 layers, first the clean layer w/ logic and behaviour. Second add the risk/what we are detecting (fraudulant transactions, credit risk repayment, insurance fraud). Finally add the messy data features

## 1.Imports

In [281]:
from __future__ import annotations
 
from datetime import date, timedelta
 
import numpy as np
import pandas as pd

## 2. Global Helpers:
1. random number generator
2. random date
3. zero-padded ID strings (adjusts on how many IDs are needed)
4. min/max values to prevent outliers such as -ve credit score 

In [282]:
def _rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)
 
 
def _random_date(rng: np.random.Generator, start: date, end: date) -> date:
    delta = (end - start).days
    return start + timedelta(days=int(rng.integers(0, delta + 1)))
 
 
def _ids(prefix: str, n: int, start: int = 1) -> list[str]:
    width = max(6, len(str(n + start)))
    return [f"{prefix}{str(i).zfill(width)}" for i in range(start, n + start)]
 
 
def _clamp(value: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, value))

## 3. Clean baseline layer

### 3.1 Shared customer generator
Customers generated for all three domains for consistency.
Columns: ID, age, income, credit_score, risk_score
Age - randomly chosen between 18 and 80
Income - log-normal distribution w/ median £35000 (ln35000 = 10.36. Realistically skeded: 10k to 500k 
Credit score - ranges 300 to 850, correlated w/ income (higher income = better credit)
Risk score - ranges 0 to 100, inverse of credit score so low credit = high risk
Noise - examples often add random noise so added this into credit/risk scores

In [283]:
def _generate_customers(n: int, rng: np.random.Generator) -> pd.DataFrame:
    
    customer_ids = _ids("CUST", n)
 
    ages = rng.integers(18, 80, size=n)
 
    log_income = rng.normal(loc=10.6, scale=0.7, size=n)  
    incomes = np.clip(np.exp(log_income), 10_000, 500_000).round(2)
 
    base_credit = 300 + (incomes / incomes.max()) * 400
    credit_noise = rng.normal(0, 50, size=n)
    credit_scores = np.clip(base_credit + credit_noise, 300, 850).astype(int)
 
    base_risk = 100 - ((credit_scores - 300) / 550) * 80
    risk_noise = rng.normal(0, 8, size=n)
    risk_scores = np.clip(base_risk + risk_noise, 0, 100).round(1)
 
    return pd.DataFrame({
        "customer_id": customer_ids,
        "age": ages,
        "income": incomes,
        "credit_score": credit_scores,
        "risk_score": risk_scores,
    })

### 3.2 Fraud clean generator
Fraud Domain - takes customers from previous and then generates accounts and transactions seperately

Account types (current 70%, saving 30%)
Transaction types (payment 65%, deposit 35%)
Transaction locations are the 7 continents but can be changed to any array, this way just for simplicity
1 Account per customer to keep simple, balance is log-normally distributed again and correlated w/ income
Transactions vary per customer and depend on:
    1. Account type (savings 50% transaction of current)
    2. Income (poisson distribution for variation) - higher income = more transactions
Transaction amount log-normally distributed and correlated w/ income (capped at 50k)
** is_fraud populated entirely as 0, Qi Wang when adding risk layer should flip a subset to 1 depending on each customer's risk score **

In [284]:
ACCOUNT_TYPES = ["current", "savings"]
TRANSACTION_TYPES = ["payment", "deposit"]
TRANSACTION_LOCATIONS = [
    "Africa", "Antarctica", "Asia", "Europe",
    "North America", "Oceania", "South America",
]
 
 
def _generate_fraud(
    customers: pd.DataFrame,
    time_period_months: int,
    rng: np.random.Generator,
) -> pd.DataFrame:
    
    n = len(customers)
    end_date = date.today()
    start_date = end_date - timedelta(days=30 * time_period_months)
 
    account_ids = _ids("ACC", n)
    account_types = rng.choice(ACCOUNT_TYPES, size=n, p=[0.70, 0.30])
 
    log_balance = np.log(customers["income"].values / 12) + rng.normal(0, 0.5, size=n)
    balances = np.clip(np.exp(log_balance), 0, 200_000).round(2)
 
    accounts = pd.DataFrame({
        "customer_id": customers["customer_id"].values,
        "account_id": account_ids,
        "account_type": account_types,
        "account_balance": balances,
    })
 
    txn_rate = np.where(account_types == "savings", 0.5, 1.0)
    income_pct = customers["income"].rank(pct=True).values
    avg_txns = np.clip((income_pct * 20 * txn_rate + 1).astype(int), 1, 40)
 
    rows = []
    txn_counter = 1
 
    for i, cust_row in customers.iterrows():
        acc = accounts.iloc[i]
        n_txns = max(1, int(rng.poisson(avg_txns[i])))
 
        for _ in range(n_txns):
            txn_date = _random_date(rng, start_date, end_date)
 
            log_amt = np.log(cust_row["income"] / 50) + rng.normal(0, 1.0)
            amount = round(float(np.clip(np.exp(log_amt), 1, 50_000)), 2)
 
            txn_type = str(rng.choice(
                TRANSACTION_TYPES,
                p=[0.65, 0.35],
            ))
 
            rows.append({
                "customer_id": cust_row["customer_id"],
                "age": cust_row["age"],
                "income": cust_row["income"],
                "risk_score": cust_row["risk_score"],
                "account_id": acc["account_id"],
                "account_type": acc["account_type"],
                "account_balance": acc["account_balance"],
                "transaction_id": f"TXN{str(txn_counter).zfill(8)}",
                "transaction_date": txn_date.isoformat(),
                "transaction_amount": amount,
                "transaction_type": txn_type,
                "transaction_location": str(rng.choice(TRANSACTION_LOCATIONS)),
                "is_fraud": 0,
            })
            txn_counter += 1
 
    return pd.DataFrame(rows)

### 3.3 Credit clean generator
Credit domain - one loan per customer, logic derived from customer's financials, repayment derived by D/E ratio

Loan amount - 0.5x to 5x annual income (capped at 200k)
Interest rate - 1.5% to 30% depending on credit score (inversely) - noise added too
Term - monthly - randomly chosen from standard options (12,24,36 etc)
Monthly payment - calculated using standard ammortisation formula (amount, rate, term)
Repayment status - takes 5 values (on-time, late1, late2, late3, defaulted)
    1. Probability depends on D/E and credit score
    2. High D/E and low credit = more likely to default
Days past due then calls on repayment status
Flag_default is set to 1 (yes) if default occurs

In [285]:
REPAYMENT_STATUSES = ["on_time", "late_1_30", "late_31_60", "late_61_90", "defaulted"]
 
 
def _generate_credit(
    customers: pd.DataFrame,
    time_period_months: int,
    rng: np.random.Generator,
) -> pd.DataFrame:
    
    n = len(customers)
    end_date = date.today()
    start_date = end_date - timedelta(days=30 * time_period_months)
 
    loan_ids = _ids("LOAN", n)
 
    multipliers = rng.uniform(0.5, 5.0, size=n)
    loan_amounts = np.clip(
        (customers["income"].values * multipliers).round(-2),
        1_000, 2_000_000,
    )
 
    base_rate = 15 - ((customers["credit_score"].values - 300) / 550) * 12
    rate_noise = rng.normal(0, 0.8, size=n)
    interest_rates = np.clip(base_rate + rate_noise, 1.5, 30.0).round(2)
 
    term_options = [12, 24, 36, 60, 120, 240, 360]
    terms = rng.choice(term_options, size=n)
 
    r = interest_rates / 100 / 12
    monthly_payments = np.where(
        r == 0,
        loan_amounts / terms,
        (loan_amounts * r) / (1 - (1 + r) ** -terms),
    ).round(2)
 
    annual_debt = monthly_payments * 12
    dti = np.clip(annual_debt / customers["income"].values, 0.01, 2.0).round(4)
 
    default_prob = np.clip(dti * 0.5 + (1 - (customers["credit_score"].values - 300) / 550) * 0.3, 0, 1)
    status_choices = []
    for prob in default_prob:
        p_on_time = max(0, 1 - prob * 2)
        p_late_1 = min(0.3, prob * 0.8)
        p_late_2 = min(0.2, prob * 0.5)
        p_late_3 = min(0.15, prob * 0.3)
        p_default = max(0, 1 - p_on_time - p_late_1 - p_late_2 - p_late_3)
        probs = np.array([p_on_time, p_late_1, p_late_2, p_late_3, p_default])
        probs /= probs.sum()
        status_choices.append(str(rng.choice(REPAYMENT_STATUSES, p=probs)))
 
    days_past_due_map = {
        "on_time": (0, 0),
        "late_1_30": (1, 30),
        "late_31_60": (31, 60),
        "late_61_90": (61, 90),
        "defaulted": (91, 720),
    }
    days_past_due = [
        int(rng.integers(*days_past_due_map[s])) if days_past_due_map[s][1] > 0
        else 0
        for s in status_choices
    ]
 
    flag_default = [1 if s == "defaulted" else 0 for s in status_choices]
 
    loan_start_dates = [
        _random_date(rng, start_date, end_date).isoformat()
        for _ in range(n)
    ]
 
    return pd.DataFrame({
        "customer_id": customers["customer_id"].values,
        "customer_income": customers["income"].values,
        "credit_score": customers["credit_score"].values,
        "risk_score": customers["risk_score"].values,
        "debt_to_income_ratio": dti,
        "loan_id": loan_ids,
        "loan_amount": loan_amounts,
        "interest_rate": interest_rates,
        "term_months": terms,
        "monthly_payment": monthly_payments,
        "loan_start_date": loan_start_dates,
        "repayment_status": status_choices,
        "days_past_due": days_past_due,
        "flag_default": flag_default,
    })

### 3.4 Insurance clean generator
Insurance domain - one policy per customer with ~40% filing a claim

Five policy types (can be changed to whatever we decide) - home 30%, auto 25%, life 205, health 15%, travel 10%
Claim reasons are what I could think of and see as examples online but are specific to each type
High fraud reasons have higher fraud probability **Use this for risk dataset **
Premium - log-normally distributed, correlated w/ income (20 - 5000)
Coverage - proprotional to premium but adds noise (max 10 million)

Claims are only generated for the 40% of customers who file one. Claim reason is taken from the policy-type- list. Claim amount is a fraction of coverage, with higher fraud reasons  producing larger fractions (30–90%) than minor ones (1–40%). Fault is assigned from the four possible values. Customers with no claim have 'none' for all claim columns 

** fraud_found populated entirely as 0, Qi Wang when adding risk layer should flip a subset to 1 depending on each customer's risk score **

In [286]:
POLICY_TYPES = ["home", "life", "auto", "health", "travel"]
CLAIM_REASONS = {
    "home":   ["fire", "flood", "theft", "accidental_damage", "subsidence"],
    "life":   ["death", "critical_illness", "disability"],
    "auto":   ["collision", "theft", "fire", "weather_damage", "vandalism"],
    "health": ["surgery", "hospitalisation", "prescription", "physiotherapy"],
    "travel": ["cancellation", "medical", "lost_luggage", "delay"],
}
FAULT_VALUES = ["policyholder", "third_party", "act_of_god", "undetermined"]
 
INSURANCE_HIGH_FRAUD_REASONS = {"theft", "collision", "fire", "cancellation", "lost_luggage"}
 
 
def _generate_insurance(
    customers: pd.DataFrame,
    time_period_months: int,
    rng: np.random.Generator,
) -> pd.DataFrame:
    
    n = len(customers)
    end_date = date.today()
    start_date = end_date - timedelta(days=30 * time_period_months)
 
    policy_ids = _ids("POL", n)
    policy_types = rng.choice(POLICY_TYPES, size=n, p=[0.30, 0.20, 0.25, 0.15, 0.10])
 
    log_premium = np.log(customers["income"].values / 100) + rng.normal(0, 0.4, size=n)
    premiums = np.clip(np.exp(log_premium), 20, 5_000).round(2)
 
    coverage_amounts = np.clip(
        premiums * rng.uniform(50, 500, size=n),
        5_000, 10_000_000,
    ).round(-2)
 
    has_claim = rng.random(size=n) < 0.40
    claim_ids = []
    claim_amounts = []
    claim_dates = []
    claim_reasons = []
    fault_values = []
    fraud_found = []
 
    claim_counter = 1
    for i in range(n):
        if has_claim[i]:
            ptype = policy_types[i]
            reason = str(rng.choice(CLAIM_REASONS[ptype]))
 
            serious = reason in {"fire", "death", "collision", "surgery"}
            frac = rng.uniform(0.3, 0.9) if serious else rng.uniform(0.01, 0.4)
            c_amount = round(float(coverage_amounts[i] * frac), 2)
 
            claim_ids.append(f"CLM{str(claim_counter).zfill(7)}")
            claim_amounts.append(c_amount)
            claim_dates.append(_random_date(rng, start_date, end_date).isoformat())
            claim_reasons.append(reason)
            fault_values.append(str(rng.choice(
                FAULT_VALUES,
                p=[0.35, 0.30, 0.20, 0.15],
            )))
            fraud_found.append(0)
            claim_counter += 1
        else:
            claim_ids.append(None)
            claim_amounts.append(None)
            claim_dates.append(None)
            claim_reasons.append(None)
            fault_values.append(None)
            fraud_found.append(None)
 
    return pd.DataFrame({
    "customer_id": customers["customer_id"].values,
    "risk_score": customers["risk_score"].values,
    "policy_id": policy_ids,
    "policy_type": policy_types,
    "premium_amount": premiums,
    "coverage_amount": coverage_amounts,
    "claim_id": claim_ids,
    "has_claim": has_claim.astype(int),
    "claim_amount": claim_amounts,
    "claim_date": claim_dates,
    "claim_reason": claim_reasons,
    "fault": fault_values,
    "fraud_found": fraud_found,
})

### 3.5 Clean generator router
Basic domain router, dictonary maping each domain to corresponding generator function.
Not sure how API will work but should be used by the API to call the correct generator depending on users input.

In [287]:
DOMAIN_GENERATORS = {
    "fraud": _generate_fraud,
    "credit": _generate_credit,
    "insurance": _generate_insurance,
}

## 4. Risk layer

### 4.1 Risk-level config
Risk Layer Configuration: Mapping user input to probability scaling.

In [288]:
RISK_MULTIPLIER = {
    "low": 0.8,
    "medium": 1.0,
    "high": 1.2,
}

### 4.2 Fraud risk function
Fraud risk layer:
This layer adds fraud labels on top of the clean fraud baseline. The main idea is to assign fraud probabilistically using the existing `risk_score`, the user-selected `risk_level`, and a few simple transaction-level signals such as transaction amount, account type, and transaction type. Higher-risk customers and more unusual transactions are more likely to be labelled as fraud.

In [289]:
def apply_risk_fraud(df: pd.DataFrame, risk_level: str, rng: np.random.Generator) -> pd.DataFrame:
    out = df.copy()

    if risk_level not in RISK_MULTIPLIER:
        raise ValueError(f"Unknown risk_level: {risk_level}. Expected one of {list(RISK_MULTIPLIER.keys())}")

    multiplier = RISK_MULTIPLIER[risk_level]

    risk_component = out["risk_score"].fillna(0).astype(float) / 100.0

    amount_reference = max(out["transaction_amount"].quantile(0.90), 1.0)
    amount_component = np.clip(out["transaction_amount"].astype(float) / amount_reference, 0, 2) / 2

    current_bonus = np.where(out["account_type"].eq("current"), 0.02, 0.0)
    payment_bonus = np.where(out["transaction_type"].eq("payment"), 0.01, 0.0)

    fraud_prob = (
        0.01
        + 0.18 * risk_component
        + 0.06 * amount_component
        + current_bonus
        + payment_bonus
    ) * multiplier

    fraud_prob = np.clip(fraud_prob, 0.0, 0.95)

    existing_flags = out["is_fraud"].fillna(0).astype(int).values
    new_flags = (rng.random(len(out)) < fraud_prob).astype(int)

    out["is_fraud"] = np.maximum(existing_flags, new_flags)

    return out

### 4.3 Credit risk function
Credit risk layer:
Starting from the clean credit baseline, this function strengthens credit risk outcomes probabilistically using risk_score, credit_score, debt_to_income_ratio, and the user-selected risk level. The original dataframe structure is preserved.


In [290]:
def apply_risk_credit(df: pd.DataFrame, risk_level: str, rng: np.random.Generator) -> pd.DataFrame:
    out = df.copy()

    if risk_level not in RISK_MULTIPLIER:
        raise ValueError(f"Unknown risk_level: {risk_level}. Expected one of {list(RISK_MULTIPLIER.keys())}")

    multiplier = RISK_MULTIPLIER[risk_level]

    risk_component = out["risk_score"].fillna(0).astype(float) / 100.0
    credit_component = 1.0 - np.clip((out["credit_score"].astype(float) - 300.0) / 550.0, 0.0, 1.0)
    dti_component = np.clip(out["debt_to_income_ratio"].fillna(0).astype(float) / 0.60, 0.0, 1.0)

    default_prob = (
        0.01
        + 0.14 * risk_component
        + 0.14 * credit_component
        + 0.18 * dti_component
    ) * multiplier
    default_prob = np.clip(default_prob, 0.0, 0.90)

    late_prob = (
        0.02
        + 0.08 * risk_component
        + 0.06 * credit_component
        + 0.10 * dti_component
    ) * multiplier
    late_prob = np.clip(late_prob, 0.0, 0.75)

    existing_default = out["flag_default"].fillna(0).astype(int).values
    sampled_default = (rng.random(len(out)) < default_prob).astype(int)
    final_default = np.maximum(existing_default, sampled_default)

    status = out["repayment_status"].astype(object).copy()
    days_past_due = out["days_past_due"].fillna(0).astype(int).copy()

    newly_defaulted = (final_default == 1) & (status != "defaulted")
    status.loc[newly_defaulted] = "defaulted"
    days_past_due.loc[newly_defaulted] = rng.integers(91, 181, size=newly_defaulted.sum())

    eligible_late = (final_default == 0) & (status == "on_time")
    sampled_late = np.zeros(len(out), dtype=bool)
    sampled_late[eligible_late] = rng.random(eligible_late.sum()) < late_prob[eligible_late]

    late_bucket_draw = np.full(len(out), -1.0)
    late_bucket_draw[sampled_late] = rng.random(sampled_late.sum())

    late_1_30_mask = sampled_late & (late_bucket_draw < 0.50)
    late_31_60_mask = sampled_late & (late_bucket_draw >= 0.50) & (late_bucket_draw < 0.85)
    late_61_90_mask = sampled_late & (late_bucket_draw >= 0.85)

    status.loc[late_1_30_mask] = "late_1_30"
    status.loc[late_31_60_mask] = "late_31_60"
    status.loc[late_61_90_mask] = "late_61_90"

    days_past_due.loc[late_1_30_mask] = rng.integers(1, 31, size=late_1_30_mask.sum())
    days_past_due.loc[late_31_60_mask] = rng.integers(31, 61, size=late_31_60_mask.sum())
    days_past_due.loc[late_61_90_mask] = rng.integers(61, 91, size=late_61_90_mask.sum())

    out["flag_default"] = final_default
    out["repayment_status"] = status
    out["days_past_due"] = days_past_due

    return out

### 4.4 Insurance risk function
Insurance risk layer:
Starting from the clean insurance baseline, this function adds insurance fraud
labels probabilistically using risk_score, user-selected risk level, and a few
simple claim-level signals. Rows without a claim remain non-applicable.

In [291]:
def apply_risk_insurance(df: pd.DataFrame, risk_level: str, rng: np.random.Generator) -> pd.DataFrame:
    out = df.copy()

    if risk_level not in RISK_MULTIPLIER:
        raise ValueError(f"Unknown risk_level: {risk_level}. Expected one of {list(RISK_MULTIPLIER.keys())}")

    multiplier = RISK_MULTIPLIER[risk_level]

    claim_mask = out["has_claim"].fillna(0).astype(int) == 1

    fraud_found = out["fraud_found"].copy()

    risk_component = out["risk_score"].fillna(0).astype(float) / 100.0

    reason_bonus = np.where(out["claim_reason"].isin(INSURANCE_HIGH_FRAUD_REASONS), 0.08, 0.0)

    fault_bonus = np.where(out["fault"].eq("policyholder"), 0.04, 0.0)

    fraud_prob = (
        0.01
        + 0.16 * risk_component
        + reason_bonus
        + fault_bonus
    ) * multiplier

    fraud_prob = np.clip(fraud_prob, 0.0, 0.90)

    existing_flags = out.loc[claim_mask, "fraud_found"].fillna(0).astype(int).values
    new_flags = (rng.random(claim_mask.sum()) < fraud_prob[claim_mask]).astype(int)
    final_flags = np.maximum(existing_flags, new_flags)

    fraud_found.loc[claim_mask] = final_flags

    fraud_found.loc[~claim_mask] = np.nan

    out["fraud_found"] = fraud_found

    return out

### 4.5 Risk router
Unified Risk Router:
This step provides a single entry point for the risk layer across all three domains.
Instead of writing separate calling logic each time, the router maps the selected domain (`fraud` `credit`, or `insurance`) to the corresponding domain-specific risk function. This keeps the implementation consistent and avoids duplicated control logic.

In [292]:
RISK_LAYER_APPLIERS = {
    "fraud": apply_risk_fraud,
    "credit": apply_risk_credit,
    "insurance": apply_risk_insurance,
}

def apply_risk_layer(
        df: pd.DataFrame,
        domain: str,
        risk_level: str,
        rng: np.random.Generator,
) -> pd.DataFrame:
    if domain not in RISK_LAYER_APPLIERS:
        raise ValueError(
            f"Unknown domain: {domain}. Expected one of {list(RISK_LAYER_APPLIERS.keys())}"
        )

    return RISK_LAYER_APPLIERS[domain](df, risk_level, rng)

## 5. Messy-data layer

### 5.1 Generic Messy Layer
This layer applies domain-agnostic data quality issues to an already generated dataset.
The first version focuses on common problems that can appear across all domains, including missing values, extra whitespace, case inconsistencies, simple string typos, and duplicate rows. The implementation avoids protected columns such as IDs and target labels, and only injects new missing values into cells that were originally non-null.

In [293]:
def apply_generic_messiness(
        df: pd.DataFrame,
        messiness_pct: float,
        rng: np.random.Generator,
) -> pd.DataFrame:
    out = df.copy()

    if not (0.0 <= messiness_pct <= 1.0):
        raise ValueError("messiness_pct must be between 0.0 and 1.0")

    n_rows = len(out)
    if n_rows == 0 or messiness_pct == 0:
        return out

    protected_columns = {
        "customer_id",
        "account_id",
        "transaction_id",
        "loan_id",
        "policy_id",
        "claim_id",
        "is_fraud",
        "flag_default",
        "fraud_found",
        "has_claim",
    }

    candidate_columns = [col for col in out.columns if col not in protected_columns]

    if not candidate_columns:
        return out

    object_columns = [col for col in candidate_columns if out[col].dtype == "object"]
    numeric_columns = [col for col in candidate_columns if pd.api.types.is_numeric_dtype(out[col])]

    # 1. Inject missing values into existing non-null cells only
    missing_cols = candidate_columns[:]
    n_missing = max(1, int(len(missing_cols) * messiness_pct))

    chosen_missing_cols = rng.choice(missing_cols, size=min(n_missing, len(missing_cols)), replace=False)

    for col in chosen_missing_cols:
        non_null_idx = out.index[out[col].notna()]
        if len(non_null_idx) == 0:
            continue

        n_cells = max(1, int(len(non_null_idx) * messiness_pct * 0.3))
        chosen_idx = rng.choice(non_null_idx, size=min(n_cells, len(non_null_idx)), replace=False)
        out.loc[chosen_idx, col] = np.nan

    # 2. Add extra whitespace to some string cells
    if object_columns:
        n_whitespace_cols = max(1, int(len(object_columns) * messiness_pct))
        chosen_ws_cols = rng.choice(object_columns, size=min(n_whitespace_cols, len(object_columns)), replace=False)

        for col in chosen_ws_cols:
            valid_idx = out.index[out[col].notna()]
            if len(valid_idx) == 0:
                continue

            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.3))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                val = out.at[idx, col]
                if isinstance(val, str):
                    out.at[idx, col] = f" {val} "

    # 3. Introduce case inconsistencies in some string cells
    if object_columns:
        n_case_cols = max(1, int(len(object_columns) * messiness_pct))
        chosen_case_cols = rng.choice(object_columns, size=min(n_case_cols, len(object_columns)), replace=False)

        for col in chosen_case_cols:
            valid_idx = out.index[out[col].notna()]
            if len(valid_idx) == 0:
                continue

            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.2))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                val = out.at[idx, col]
                if isinstance(val, str):
                    mode = rng.choice(["upper", "lower", "title"])
                    if mode == "upper":
                        out.at[idx, col] = val.upper()
                    elif mode == "lower":
                        out.at[idx, col] = val.lower()
                    else:
                        out.at[idx, col] = val.title()

    # 4. Inject simple string typos by replacing one character
    if object_columns:
        n_typo_cols = max(1, int(len(object_columns) * messiness_pct))
        chosen_typo_cols = rng.choice(object_columns, size=min(n_typo_cols, len(object_columns)), replace=False)

        alphabet = list("abcdefghijklmnopqrstuvwxyz")

        for col in chosen_typo_cols:
            valid_idx = out.index[out[col].notna()]
            if len(valid_idx) == 0:
                continue

            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.2))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                val = out.at[idx, col]
                if isinstance(val, str) and len(val) >= 3:
                    pos = rng.integers(0, len(val))
                    replacement = rng.choice(alphabet)
                    out.at[idx, col] = val[:pos] + replacement + val[pos + 1:]

    # 5. Add a small number of duplicate rows
    n_dup = max(1, int(n_rows * messiness_pct * 0.2))
    dup_idx = rng.choice(out.index, size=min(n_dup, n_rows), replace=False)
    dup_rows = out.loc[dup_idx].copy()

    out = pd.concat([out, dup_rows], ignore_index=True)

    return out

### 5.2 Fraud-specific messy function
This layer adds fraud-domain data quality issues on top of the generic messy output.
The first version focuses on transaction-level label inconsistencies, location variations, duplicate
transaction rows, and a small number of unrealistic account balances. The aim is to introduce
fraud-related messy patterns without changing the overall dataframe structure.

In [294]:
def apply_fraud_business_errors(
        df: pd.DataFrame,
        messiness_pct: float,
        rng: np.random.Generator,
) -> pd.DataFrame:
    out = df.copy()

    if not (0.0 <= messiness_pct <= 1.0):
        raise ValueError("messiness_pct must be between 0.0 and 1.0")

    n_rows = len(out)
    if n_rows == 0 or messiness_pct == 0:
        return out

    # Fraud-specific business anomaly: unrealistic account balances
    if "account_balance" in out.columns:
        valid_idx = out.index[out["account_balance"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.1))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                val = out.at[idx, "account_balance"]
                if pd.notna(val):
                    out.at[idx, "account_balance"] = -abs(float(val))

    return out

### 5.3 Credit-specific messy function
This layer adds credit-domain data quality issues on top of the generic messy output.
The first version focuses on interest-rate formatting inconsistencies, repayment-status label variation, a small number of unrealistic income values, and mild inconsistencies between `days_past_due` and `repayment_status`.

In [295]:
def apply_credit_business_errors(
        df: pd.DataFrame,
        messiness_pct: float,
        rng: np.random.Generator,
) -> pd.DataFrame:
    out = df.copy()

    if not (0.0 <= messiness_pct <= 1.0):
        raise ValueError("messiness_pct must be between 0.0 and 1.0")

    n_rows = len(out)
    if n_rows == 0 or messiness_pct == 0:
        return out

    # 1. Interest-rate formatting inconsistencies
    if "interest_rate" in out.columns:
        out["interest_rate"] = out["interest_rate"].astype(object)
        valid_idx = out.index[out["interest_rate"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.2))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                val = out.at[idx, "interest_rate"]
                if pd.notna(val):
                    x = float(val)
                    variants = [x, f"{x:.2f}%", f"{x / 100:.4f}", f" {x:.1f}% "]
                    out.at[idx, "interest_rate"] = rng.choice(variants)

    # 2. Repayment-status label inconsistencies
    if "repayment_status" in out.columns:
        valid_idx = out.index[out["repayment_status"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.2))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            typo_map = {
                "on_time": ["ON_TIME", "on time", "On_Time"],
                "late_1_30": ["LATE_1_30", "late 1 30", "Late_1_30"],
                "late_31_60": ["LATE_31_60", "late 31 60", "Late_31_60"],
                "late_61_90": ["LATE_61_90", "late 61 90", "Late_61_90"],
                "defaulted": ["DEFAULTED", "default", "Defau1ted"],
            }

            for idx in chosen_idx:
                val = out.at[idx, "repayment_status"]
                if isinstance(val, str) and val in typo_map:
                    out.at[idx, "repayment_status"] = rng.choice(typo_map[val])

    #  3. Introduce a few unrealistic customer incomes
    if "customer_income" in out.columns:
        valid_idx = out.index[out["customer_income"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.1))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                val = out.at[idx, "customer_income"]
                if pd.notna(val):
                    mode = rng.choice(["negative", "zero", "scaled_down"])
                    if mode == "negative":
                        out.at[idx, "customer_income"] = -abs(float(val))
                    elif mode == "zero":
                        out.at[idx, "customer_income"] = 0
                    else:
                        out.at[idx, "customer_income"] = round(float(val) * 0.1, 2)

    #  4. Introduce mild inconsistency between days_past_due and repayment_status
    if {"days_past_due", "repayment_status"}.issubset(out.columns):
        valid_idx = out.index[out["days_past_due"].notna() & out["repayment_status"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.1))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                status = out.at[idx, "repayment_status"]
                if isinstance(status, str):
                    if "on_time" in status.lower().replace(" ", "_"):
                        out.at[idx, "days_past_due"] = rng.integers(5, 25)
                    elif "default" in status.lower():
                        out.at[idx, "days_past_due"] = rng.integers(0, 20)

    return out

### 5.4 Insurance-specific messy function
This layer adds insurance-domain data quality issues on top of the generic messy output.
The first version focuses on hidden missing values, label inconsistencies in `claim_reason` and `fault`, and a small number of inconsistencies between `claim_amount` and `coverage_amount`.

In [296]:
def apply_insurance_business_errors(
        df: pd.DataFrame,
        messiness_pct: float,
        rng: np.random.Generator,
) -> pd.DataFrame:
    out = df.copy()

    if not (0.0 <= messiness_pct <= 1.0):
        raise ValueError("messiness_pct must be between 0.0 and 1.0")

    n_rows = len(out)
    if n_rows == 0 or messiness_pct == 0:
        return out

    claim_mask = out["has_claim"].fillna(0).astype(int) == 1

    # 1. Hidden missing values in claim-related text fields
    for col in ["claim_reason", "fault"]:
        if col in out.columns:
            valid_idx = out.index[claim_mask & out[col].notna()]
            if len(valid_idx) > 0:
                n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.15))
                chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)
                for idx in chosen_idx:
                    out.at[idx, col] = rng.choice(["none", "NONE", " None "])

    # 2. Label inconsistencies / misspellings in claim_reason
    if "claim_reason" in out.columns:
        valid_idx = out.index[claim_mask & out["claim_reason"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.20))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            typo_map = {
                "collision": ["COLLISION", "collision ", "collison"],
                "theft": ["THEFT", " theft ", "thft"],
                "fire": ["FIRE", " fire ", "fiire"],
                "cancellation": ["CANCELLATION", " cancellation ", "cancelation"],
                "lost_luggage": ["LOST_LUGGAGE", "lost luggage", "lost_lugage"],
                "illness": ["ILLNESS", " illness ", "illnes"],
                "medical": ["MEDICAL", " medical ", "medcal"],
                "weather": ["WEATHER", " weather ", "weathr"],
            }

            for idx in chosen_idx:
                val = out.at[idx, "claim_reason"]
                if isinstance(val, str) and val in typo_map:
                    out.at[idx, "claim_reason"] = rng.choice(typo_map[val])

    # 3. Label inconsistencies / misspellings in fault
    if "fault" in out.columns:
        valid_idx = out.index[claim_mask & out["fault"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.15))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            fault_map = {
                "policyholder": ["POLICYHOLDER", "policy holder", "polcyholder"],
                "third_party": ["THIRD_PARTY", "third party", "third_prty"],
                "act_of_god": ["ACT_OF_GOD", "act of god", "act_of_g0d"],
            }

            for idx in chosen_idx:
                val = out.at[idx, "fault"]
                if isinstance(val, str) and val in fault_map:
                    out.at[idx, "fault"] = rng.choice(fault_map[val])

    # 4. Mild inconsistency between claim_amount and coverage_amount
    if {"claim_amount", "coverage_amount"}.issubset(out.columns):
        valid_idx = out.index[claim_mask & out["claim_amount"].notna() & out["coverage_amount"].notna()]
        if len(valid_idx) > 0:
            n_cells = max(1, int(len(valid_idx) * messiness_pct * 0.10))
            chosen_idx = rng.choice(valid_idx, size=min(n_cells, len(valid_idx)), replace=False)

            for idx in chosen_idx:
                coverage = out.at[idx, "coverage_amount"]
                if pd.notna(coverage):
                    out.at[idx, "claim_amount"] = round(float(coverage) * rng.uniform(1.05, 1.30), 2)

    return out

### 5.5 Messiness-level config
Global mapping from user-selected messiness level to the percentage of rows/cells affected by the generic and domain-specific messy-data layers.

In [297]:
MESSINESS_PCT = {
    "low": 0.05,
    "medium": 0.10,
    "high": 0.20,
}

### 5.6 Messy router
Unified Messy Router:This step provides a single entry point for the messy-data layer across all
three domains.
The messy layer is applied in two stages. First, a generic function introduces domain-agnostic data quality issues such as missing values, whitespace, typos, and duplicate rows. Then, a domain-specific function adds business-like errors that are more closely related to fraud, credit, or insurance records. This keeps the implementation modular and avoids duplicated control logic.

In [298]:
MESSY_DOMAIN_APPLIERS = {
    "fraud": apply_fraud_business_errors,
    "credit": apply_credit_business_errors,
    "insurance": apply_insurance_business_errors,
}


def apply_messy_layer(
        df: pd.DataFrame,
        domain: str,
        messiness_pct: float,
        rng_generic: np.random.Generator,
        rng_domain: np.random.Generator,
) -> pd.DataFrame:
    if domain not in MESSY_DOMAIN_APPLIERS:
        raise ValueError(
            f"Unknown domain: {domain}. Expected one of {list(MESSY_DOMAIN_APPLIERS.keys())}"
        )

    # apply the generic messy layer
    out = apply_generic_messiness(df, messiness_pct=messiness_pct, rng=rng_generic)

    # apply the domain-specific messy layer
    out = MESSY_DOMAIN_APPLIERS[domain](out, messiness_pct=messiness_pct, rng=rng_domain)

    return out

## 6. Final pipeline

In [299]:
def build_final_dataset(
        domain: str,
        n_customers: int,
        time_period_months: int,
        risk_level: str,
        messiness_level: str,
        seed: int,
) -> pd.DataFrame:

    if domain not in DOMAIN_GENERATORS:
        raise ValueError(f"Unknown domain: {domain}. Expected one of {list(DOMAIN_GENERATORS.keys())}")

    if risk_level not in RISK_MULTIPLIER:
        raise ValueError(f"Unknown risk_level: {risk_level}. Expected one of {list(RISK_MULTIPLIER.keys())}")

    if messiness_level not in MESSINESS_PCT:
        raise ValueError(
            f"Unknown messiness_level: {messiness_level}. Expected one of {list(MESSINESS_PCT.keys())}"
        )

    if n_customers <= 0:
        raise ValueError("n_customers must be greater than 0")

    if time_period_months <= 0:
        raise ValueError("time_period_months must be greater than 0")

    clean_rng = _rng(seed)
    risk_rng = _rng(seed + 1)
    messy_generic_rng = _rng(seed + 2)
    messy_domain_rng = _rng(seed + 3)

    # 1: generate the shared customer baseline
    customers_df = _generate_customers(n_customers, clean_rng)

    # 2: generate the clean domain-specific baseline
    clean_df = DOMAIN_GENERATORS[domain](customers_df, time_period_months, clean_rng)

    # 3: apply the risk layer
    risk_df = apply_risk_layer(clean_df, domain=domain, risk_level=risk_level, rng=risk_rng)

    # 4: map messiness level to a percentage
    messiness_pct = MESSINESS_PCT[messiness_level]

    # 5: apply the messy-data layer
    final_df = apply_messy_layer(
        risk_df,
        domain=domain,
        messiness_pct=messiness_pct,
        rng_generic=messy_generic_rng,
        rng_domain=messy_domain_rng,
    )

    return final_df

## 7. Validation / tests

### 7.1 End-to-end test of the final dataset builder.

In [300]:
fraud_final = build_final_dataset(
    domain="fraud",
    n_customers=20,
    time_period_months=12,
    risk_level="high",
    messiness_level="medium",
    seed=42,
)

credit_final = build_final_dataset(
    domain="credit",
    n_customers=20,
    time_period_months=12,
    risk_level="high",
    messiness_level="medium",
    seed=42,
)

insurance_final = build_final_dataset(
    domain="insurance",
    n_customers=20,
    time_period_months=12,
    risk_level="high",
    messiness_level="medium",
    seed=42,
)

print("Fraud final shape:", fraud_final.shape)
print("Credit final shape:", credit_final.shape)
print("Insurance final shape:", insurance_final.shape)

Fraud final shape: (210, 13)
Credit final shape: (21, 14)
Insurance final shape: (21, 13)


### 7.2 Final sanity checks

In [301]:
assert len(fraud_final) > 0
assert len(credit_final) > 0
assert len(insurance_final) > 0

assert "is_fraud" in fraud_final.columns
assert "flag_default" in credit_final.columns
assert "fraud_found" in insurance_final.columns

assert "risk_score" in credit_final.columns
assert "risk_score" in insurance_final.columns
assert "has_claim" in insurance_final.columns

# Insurance rows without claims should still have non-applicable fraud labels.
assert insurance_final.loc[insurance_final["has_claim"] == 0, "fraud_found"].isna().all()

print("Final sanity checks passed.")

Final sanity checks passed.


### 7.3 Reproducibility check

In [302]:
fraud_final_repeat = build_final_dataset(
    domain="fraud",
    n_customers=20,
    time_period_months=12,
    risk_level="high",
    messiness_level="medium",
    seed=42,
)

assert fraud_final.equals(fraud_final_repeat)

print("Reproducibility check passed.")

Reproducibility check passed.


## 8. Simple export helper for final datasets(.csv version).

In [303]:
def export_dataset(df: pd.DataFrame, filepath: str) -> None:
    if filepath.lower().endswith(".csv"):
        df.to_csv(filepath, index=False)
    elif filepath.lower().endswith(".xlsx"):
        df.to_excel(filepath, index=False)
    else:
        raise ValueError("Unsupported file format. Use .csv or .xlsx")

In [304]:
export_dataset(fraud_final, "fraud_final.csv")
export_dataset(credit_final, "credit_final.csv")
export_dataset(insurance_final, "insurance_final.csv")

print("Export completed.")

Export completed.
